# IDS Project — Notebook 3: Real-Time Detection
**Pipeline:** Scapy → CICFlowMeter → XGBoost → Live Dashboard  
> ⚠️ Run this notebook as **Administrator** — raw packet capture requires elevated privileges.  
> Install [Npcap](https://npcap.com/#download) before running.

## 1. Imports

In [4]:
import numpy as np
import pandas as pd
import pickle
import threading
import queue
import time
import warnings
from datetime import datetime
from collections import deque
warnings.filterwarnings('ignore')

import xgboost as xgb

# Live dashboard
import ipywidgets as widgets
from IPython.display import display, clear_output

# Packet capture
from scapy.all import sniff, IP, TCP, UDP, get_if_list

# Flow feature extraction
from cicflowmeter.flow_session import FlowSession

print('All imports OK.')

All imports OK.


## 2. Load Models & Artifacts

In [5]:
# Load XGBoost models
xgb_binary = xgb.XGBClassifier()
xgb_binary.load_model('xgb_binary.json')

xgb_multi = xgb.XGBClassifier()
xgb_multi.load_model('xgb_multi.json')

# Load preprocessing artifacts
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

with open('selected_features.pkl', 'rb') as f:
    selected_features = pickle.load(f)

print(f'Models loaded.')
print(f'Attack classes: {le.classes_}')
print(f'Features used: {len(selected_features)}')

Models loaded.
Attack classes: ['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']
Features used: 30


## 3. Select Network Interface

In [6]:
# # List available interfaces
# interfaces = get_if_list()
# print('Available network interfaces:')
# for i, iface in enumerate(interfaces):
#     print(f'  [{i}] {iface}')

Available network interfaces:
  [0] \Device\NPF_{1A03D685-7283-4175-B1F1-D280E1C01A77}
  [1] \Device\NPF_{8D3468AB-EA2D-48E8-A367-00F005A5A27F}
  [2] \Device\NPF_{6594FA6D-ED57-4882-B2D4-6E9C5C3F10EB}
  [3] \Device\NPF_{B63D0737-D58B-4719-AC4B-302707E0A1E7}
  [4] \Device\NPF_{D1DEABA3-F5FF-4B20-9BAC-B6D1C5C052A2}
  [5] \Device\NPF_{E8A0D663-743E-480A-98A5-D80AF9EDB2ED}
  [6] \Device\NPF_{C6B8375B-FA1C-49ED-B6F6-18FD251669C7}
  [7] \Device\NPF_{83F04586-6DCD-415D-ACEE-B455D3012108}
  [8] \Device\NPF_{B1E0961D-0C84-4B3C-991D-ABC10AC436D8}
  [9] \Device\NPF_{52264893-DF2B-432C-AEA8-84961D34A4D7}
  [10] \Device\NPF_{892AB3D4-EB8C-4405-89E6-FFDF697609DB}
  [11] \Device\NPF_Loopback
  [12] \Device\NPF_{16E8299B-04C6-4402-AB67-4789587BA2EE}


In [8]:
from scapy.all import conf

print('Available network interfaces:')
for i, iface in enumerate(conf.ifaces.values()):
    ip = iface.ip if iface.ip else "No IP"
    print(f'  [{i}] {iface.name} | {iface.description} | {ip}')

Available network interfaces:
  [0] Local Area Connection* 8 | WAN Miniport (Network Monitor) | No IP
  [1] Local Area Connection* 7 | WAN Miniport (IPv6) | No IP
  [2] Local Area Connection* 6 | WAN Miniport (IP) | No IP
  [3] Cellular | DW5930e-eSIM Snapdragon X55 5G | 169.254.57.149
  [4] Bluetooth Network Connection | Bluetooth Device (Personal Area Network) | 169.254.200.193
  [5] Wi-Fi | Intel(R) Wi-Fi 6E AX211 160MHz | 10.221.0.14
  [6] VMware Network Adapter VMnet8 | VMware Virtual Ethernet Adapter for VMnet8 | 192.168.87.1
  [7] VMware Network Adapter VMnet1 | VMware Virtual Ethernet Adapter for VMnet1 | 192.168.75.1
  [8] Local Area Connection* 10 | Microsoft Wi-Fi Direct Virtual Adapter #2 | 169.254.193.104
  [9] Local Area Connection* 9 | Microsoft Wi-Fi Direct Virtual Adapter | 169.254.44.250
  [10] Ethernet 2 | VirtualBox Host-Only Ethernet Adapter | 192.168.56.1
  [11] Loopback Pseudo-Interface 1 | Software Loopback Interface 1 | 127.0.0.1
  [12] Ethernet | Intel(R) Ethe

In [9]:
# Set the interface index you want to monitor
# Usually 0 = loopback, 1 = main Ethernet/WiFi
INTERFACE_INDEX = 11
INTERFACE = interfaces[INTERFACE_INDEX]
print(f'Monitoring: {INTERFACE}')

Monitoring: \Device\NPF_Loopback


## 4. Feature Extraction & Prediction Engine

In [10]:
class IDSEngine:
    """Captures packets, extracts flow features, runs model inference."""

    def __init__(self, interface, models, scaler, le, features, max_alerts=200):
        self.interface = interface
        self.xgb_binary = models['binary']
        self.xgb_multi  = models['multi']
        self.scaler = scaler
        self.le = le
        self.features = features

        # Shared state (thread-safe)
        self.alerts = deque(maxlen=max_alerts)
        self.stats = {
            'total_flows': 0,
            'total_attacks': 0,
            'packets_captured': 0,
        }
        self.lock = threading.Lock()
        self._stop_event = threading.Event()

        # CICFlowMeter flow session
        self.flow_session = FlowSession()
        self.flow_session.packets_count = 0

    def _predict_flow(self, flow_features: dict):
        """Extract selected features, scale, predict."""
        row = []
        for feat in self.features:
            val = flow_features.get(feat, 0.0)
            try:
                val = float(val)
            except (ValueError, TypeError):
                val = 0.0
            if not np.isfinite(val):
                val = 0.0
            row.append(val)

        X = np.array(row).reshape(1, -1)
        X_scaled = self.scaler.transform(X)

        is_attack = bool(self.xgb_binary.predict(X_scaled)[0])
        confidence = float(self.xgb_binary.predict_proba(X_scaled)[0][1])

        attack_type = 'BENIGN'
        if is_attack:
            label_idx = self.xgb_multi.predict(X_scaled)[0]
            attack_type = self.le.inverse_transform([label_idx])[0]

        return is_attack, attack_type, confidence

    def _process_packet(self, pkt):
        with self.lock:
            self.stats['packets_captured'] += 1

        # Feed packet into flow session
        self.flow_session.on_packet_received(pkt)

        # Check for completed flows
        completed = self.flow_session.get_flows()
        for flow in completed:
            flow_dict = flow.to_dict()
            is_attack, attack_type, confidence = self._predict_flow(flow_dict)

            with self.lock:
                self.stats['total_flows'] += 1
                if is_attack:
                    self.stats['total_attacks'] += 1
                    src_ip = flow_dict.get('Src IP', '?')
                    dst_ip = flow_dict.get('Dst IP', '?')
                    dst_port = flow_dict.get('Dst Port', '?')
                    self.alerts.appendleft({
                        'time':       datetime.now().strftime('%H:%M:%S'),
                        'src_ip':     src_ip,
                        'dst_ip':     dst_ip,
                        'dst_port':   dst_port,
                        'attack':     attack_type,
                        'confidence': f'{confidence:.1%}',
                    })

    def start(self):
        self._stop_event.clear()
        self._thread = threading.Thread(
            target=sniff,
            kwargs={
                'iface': self.interface,
                'prn': self._process_packet,
                'store': False,
                'stop_filter': lambda _: self._stop_event.is_set(),
            },
            daemon=True
        )
        self._thread.start()
        print(f'IDS started on {self.interface}')

    def stop(self):
        self._stop_event.set()
        print('IDS stopped.')

    def get_snapshot(self):
        with self.lock:
            return dict(self.stats), list(self.alerts)


engine = IDSEngine(
    interface=INTERFACE,
    models={'binary': xgb_binary, 'multi': xgb_multi},
    scaler=scaler,
    le=le,
    features=selected_features
)

print('IDS Engine ready.')

RuntimeError: no output_mode provided

## 5. Live Dashboard

In [ ]:
# ── Dashboard Layout ──────────────────────────────────────────────────────────

title = widgets.HTML(
    value='<h2 style="color:#1a73e8; font-family:monospace;">🛡 Network IDS — Live Monitor</h2>'
)

# Stats row
stat_packets  = widgets.HTML()
stat_flows    = widgets.HTML()
stat_attacks  = widgets.HTML()
stat_rate     = widgets.HTML()
stats_box = widgets.HBox([stat_packets, stat_flows, stat_attacks, stat_rate])

# Alert table
alert_out = widgets.Output(
    layout=widgets.Layout(height='400px', overflow_y='scroll',
                          border='1px solid #ddd', padding='8px')
)

# Controls
btn_start = widgets.Button(description='▶ Start', button_style='success',
                           layout=widgets.Layout(width='120px'))
btn_stop  = widgets.Button(description='■ Stop',  button_style='danger',
                           layout=widgets.Layout(width='120px'))
btn_clear = widgets.Button(description='⌫ Clear', button_style='warning',
                           layout=widgets.Layout(width='120px'))
status_label = widgets.Label(value='Status: stopped')
controls = widgets.HBox([btn_start, btn_stop, btn_clear, status_label])

dashboard = widgets.VBox([title, controls, stats_box, alert_out])


def _stat_card(label, value, color='#1a73e8'):
    return (f'<div style="background:#f8f9fa; border-left:4px solid {color}; '
            f'padding:8px 16px; margin:4px; border-radius:4px; min-width:150px;">'
            f'<div style="font-size:11px; color:#666;">{label}</div>'
            f'<div style="font-size:22px; font-weight:bold; color:{color};">{value}</div></div>')


def _render_alerts(alerts):
    if not alerts:
        return '<p style="color:#888; font-family:monospace;">No attacks detected yet.</p>'
    rows = ''.join(
        f'<tr style="background:{"#fff3cd" if i % 2 == 0 else "#ffe0b2"};">' +
        f'<td style="padding:4px 8px;">{a["time"]}</td>' +
        f'<td style="padding:4px 8px; color:#d32f2f; font-weight:bold;">{a["attack"]}</td>' +
        f'<td style="padding:4px 8px;">{a["src_ip"]}</td>' +
        f'<td style="padding:4px 8px;">{a["dst_ip"]}:{a["dst_port"]}</td>' +
        f'<td style="padding:4px 8px;">{a["confidence"]}</td>' +
        '</tr>'
        for i, a in enumerate(alerts)
    )
    return (
        '<table style="width:100%; font-family:monospace; font-size:12px; border-collapse:collapse;">'
        '<thead><tr style="background:#1a73e8; color:white;">'
        '<th style="padding:6px;">Time</th><th>Attack Type</th>'
        '<th>Source IP</th><th>Destination</th><th>Confidence</th>'
        '</tr></thead><tbody>' + rows + '</tbody></table>'
    )


_refresh_running = False


def _refresh_loop():
    global _refresh_running
    while _refresh_running:
        stats, alerts = engine.get_snapshot()
        attack_rate = (
            f"{stats['total_attacks'] / stats['total_flows']:.1%}"
            if stats['total_flows'] > 0 else '0.0%'
        )
        stat_packets.value  = _stat_card('Packets',      stats['packets_captured'], '#1a73e8')
        stat_flows.value    = _stat_card('Flows',        stats['total_flows'],      '#0f9d58')
        stat_attacks.value  = _stat_card('Attacks',      stats['total_attacks'],    '#d32f2f')
        stat_rate.value     = _stat_card('Attack Rate',  attack_rate,               '#f4b400')

        with alert_out:
            clear_output(wait=True)
            display(widgets.HTML(_render_alerts(alerts)))
        time.sleep(1.0)


def on_start(_):
    global _refresh_running
    engine.start()
    status_label.value = f'Status: MONITORING {INTERFACE}'
    _refresh_running = True
    threading.Thread(target=_refresh_loop, daemon=True).start()


def on_stop(_):
    global _refresh_running
    _refresh_running = False
    engine.stop()
    status_label.value = 'Status: stopped'


def on_clear(_):
    with engine.lock:
        engine.alerts.clear()
        engine.stats['total_attacks'] = 0
        engine.stats['total_flows'] = 0
        engine.stats['packets_captured'] = 0


btn_start.on_click(on_start)
btn_stop.on_click(on_stop)
btn_clear.on_click(on_clear)

display(dashboard)
print('Dashboard ready. Press ▶ Start to begin monitoring.')

---
## 6. Manual Test — Simulate Attack Flow
Use this cell to verify the model works **without needing live traffic** (for demo/presentation).

In [ ]:
# ── Simulate a DDoS-like flow for demo purposes ───────────────────────────────
# Load one real sample from the test set and inject it directly

X_test  = np.load('X_test.npy')
y_test_m = np.load('y_test_multi.npy')

# Pick a confirmed attack sample
y_test_b = np.load('y_test_binary.npy')
attack_indices = np.where(y_test_b == 1)[0]

print(f'Simulating {min(5, len(attack_indices))} attack flows from test set:')
print()

results = []
for idx in attack_indices[:5]:
    X_row = X_test[idx].reshape(1, -1)  # already scaled

    is_attack_pred = bool(xgb_binary.predict(X_row)[0])
    confidence     = float(xgb_binary.predict_proba(X_row)[0][1])
    attack_type    = le.inverse_transform(xgb_multi.predict(X_row))[0]
    true_label     = le.inverse_transform([y_test_m[idx]])[0]

    results.append({
        'True Label':       true_label,
        'Predicted':        attack_type if is_attack_pred else 'BENIGN',
        'Confidence':       f'{confidence:.1%}',
        'Correct':          '✓' if true_label == (attack_type if is_attack_pred else 'BENIGN') else '✗'
    })

pd.DataFrame(results)

---
## Architecture Summary

```
Network Interface (Npcap)
        │
     Scapy sniff()
        │
  CICFlowMeter (flow session)
        │  builds 80-feature flows
  Select top-30 features
        │
  StandardScaler (saved from training)
        │
  XGBoost Binary ──► BENIGN / ATTACK
        │                   │
        │            XGBoost Multi-class
        │                   │
        └───────────► Attack Type
                       (DDoS, PortScan, etc.)
                            │
                  ipywidgets Live Dashboard
```